## Libraries

In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.neighbors import NearestNeighbors

import math
import torch
import torch.nn as nn

## Config

In [ ]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_END_DATE  = pd.Timestamp("2022-03-31")
TEST_START_DATE = pd.Timestamp("2022-04-01")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- BEST LAG SET & BEST RFF-KERNEL PARAMS (from tuning) ----
best_lag_set = [1, 2, 3, 12]  # you had (1,2,3,12) in the screenshot; add 24 if you want
best_params = {
    "gamma_spatial": 1.0,
    "gamma_local":   0.1,
    "gamma_macro":   0.05,
    "gamma_lag":     0.1,
    "hidden_dim":    128,
    "weight_decay":  1e-4,
}

# RFF feature dimensions per group (you can tweak these)
RFF_DIMS = {
    "spatial": 128,
    "local":   256,
    "macro":   64,
    "lag":     512,
}

# ---- FEATURES ----
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

Using device: cuda


## Metric functions

In [3]:
def mae(y,yhat): return np.mean(np.abs(y-yhat))
def rmse(y,yhat): return np.sqrt(np.mean((y-yhat)**2))
def smape(y,yhat,eps=1e-8):
    return 100*np.mean(2*np.abs(yhat-y)/(np.abs(y)+np.abs(yhat)+eps))

def mase(y,yhat,y_train,m=12,eps=1e-8):
    naive = np.abs(y_train[m:] - y_train[:-m])
    return np.mean(np.abs(y-yhat)) / (np.mean(naive)+eps)

def directional_accuracy(df,entity,time,y,yhat):
    def f(x):
        return np.mean(np.sign(x[y].diff()) == np.sign(x[yhat].diff()))
    return df.groupby(entity).apply(f).mean()

def growth_rate_error(df,entity,time,y,yhat,m=12):
    def f(x):
        return np.mean(np.abs((x[y].pct_change(m) - x[yhat].pct_change(m))))
    return df.groupby(entity).apply(f).mean()

def morans_i(residuals,xs,ys,k=5,eps=1e-8):
    residuals = np.asarray(residuals)
    N = len(residuals)
    X = residuals - np.mean(residuals)
    coords = np.column_stack([xs,ys])
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(coords)
    distances, idx = nbrs.kneighbors(coords)

    # inverse-distance weights
    W = np.zeros((N,N))
    for i in range(N):
        neigh_idx = idx[i,1:]
        w = 1.0 / (distances[i,1:] + eps)
        W[i,neigh_idx] = w

    # row standardise
    row_sums = W.sum(axis=1, keepdims=True) + eps
    W = W / row_sums

    S0 = W.sum()
    num = np.sum(W*(X[:,None]*X[None,:]))
    den = np.sum(X**2) + eps
    return (N/S0)*num/den

def crps_gaussian(y,mu,sigma,eps=1e-8):
    y   = np.asarray(y)
    mu  = np.asarray(mu)
    sig = np.asarray(sigma) + eps
    a = (y-mu)/sig
    return np.mean(sig*(1/np.sqrt(np.pi) - 2*norm.pdf(a) - a*(2*norm.cdf(a)-1)))


## RFF SVR approx kernel model

In [4]:
class RFFBlock(nn.Module):
    """
    x -> sqrt(2/m) * cos(Wx + b), W ~ N(0, 2*gamma), b ~ U(0,2π)
    """
    def __init__(self, input_dim, output_dim, gamma):
        super().__init__()
        self.output_dim = output_dim
        self.W = nn.Parameter(
            torch.randn(output_dim, input_dim) * math.sqrt(2*gamma),
            requires_grad=False
        )
        self.b = nn.Parameter(
            torch.rand(output_dim) * 2*math.pi,
            requires_grad=False
        )
        self.scale = math.sqrt(2.0/output_dim)

    def forward(self, x):
        return torch.cos(x @ self.W.T + self.b) * self.scale

class CompositeRFF(nn.Module):
    def __init__(self, group_dims, output_dims, gammas):
        super().__init__()
        self.blocks = nn.ModuleList([
            RFFBlock(gdim, odim, gamma)
            for gdim, odim, gamma in zip(group_dims, output_dims, gammas)
        ])

    def forward(self, groups):
        mapped = [block(g) for block, g in zip(self.blocks, groups)]
        return torch.cat(mapped, dim=1)

class RFFKernelRegressor(nn.Module):
    def __init__(self, group_dims, output_dims, gammas, hidden_dim=128):
        super().__init__()
        self.rff = CompositeRFF(group_dims, output_dims, gammas)
        total_rff_dim = sum(output_dims)
        self.mlp = nn.Sequential(
            nn.Linear(total_rff_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, groups):
        z = self.rff(groups)
        return self.mlp(z).squeeze(-1)

def train_rff_model(
    model,
    train_groups,
    y_train_scaled,
    val_groups,   # here val_groups == test_groups for final model, but we still use early stopping
    y_val_scaled,
    epochs=80,
    lr=1e-3,
    weight_decay=0.0,
    patience=8,
):
    device = next(model.parameters()).device
    y_train = torch.tensor(y_train_scaled, dtype=torch.float32, device=device)
    y_val   = torch.tensor(y_val_scaled,   dtype=torch.float32, device=device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    loss_fn = nn.MSELoss()

    best_val_loss = float("inf")
    best_val_pred = None
    epochs_no_improve = 0

    for _ in range(epochs):
        # train step
        model.train()
        optimizer.zero_grad()
        y_pred_train = model(train_groups)
        loss = loss_fn(y_pred_train, y_train)
        loss.backward()
        optimizer.step()

        # validation step
        model.eval()
        with torch.no_grad():
            y_pred_val = model(val_groups)
            val_loss = loss_fn(y_pred_val, y_val).item()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_val_pred = y_pred_val.detach().cpu().numpy()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    return best_val_pred


## Feature grouping

In [5]:
spatial_cont = ["centroid_x","centroid_y","CoL_distance_km"]
local_struct = [
    "AverageNeighbourPrice","local_I","area_km2","LA_FE",
    "dwelling_stock","population","rail_station_entry_exit",
    "claimant_count_prop","planning_decisions_per_1000",
    "planning_granted_prop"
]
macro = ["sdlt_perc_threshold","ashe_weekly","base_rate","GDP","CPIH"]

def build_group_indices(all_feature_cols):
    def idx(names):
        return [all_feature_cols.index(c) for c in names if c in all_feature_cols]
    spatial_idx = idx(spatial_cont)
    local_idx   = idx(local_struct)
    macro_idx   = idx(macro)
    lag_idx     = [i for i,c in enumerate(all_feature_cols) if c.startswith("stl_")]
    return spatial_idx, local_idx, macro_idx, lag_idx

def build_groups_tensors(X, group_indices):
    X_t = torch.tensor(X.values, dtype=torch.float32, device=device)
    return [X_t[:, idx] for idx in group_indices]


## Load data

In [6]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL,TIME_COL]).reset_index(drop=True)

df_train_all = df[df[TIME_COL] <= TRAIN_END_DATE].copy()
df_test      = df[df[TIME_COL] >= TEST_START_DATE].copy()

## Training

In [7]:
df_train_all["stl_trend"]=np.nan
df_train_all["stl_seasonal"]=np.nan
df_train_all["stl_resid"]=np.nan

for la,sub in df_train_all.groupby(ENTITY_COL):
    sub=sub.sort_values(TIME_COL)
    if len(sub)<24: 
        continue
    stl=STL(sub[TARGET_COL],period=12,robust=True).fit()
    df_train_all.loc[sub.index,"stl_trend"]=stl.trend
    df_train_all.loc[sub.index,"stl_seasonal"]=stl.seasonal
    df_train_all.loc[sub.index,"stl_resid"]=stl.resid

# Extend STL into test
df_test["stl_trend"]=np.nan
df_test["stl_seasonal"]=np.nan
df_test["stl_resid"]=0.0

for la in df_train_all[ENTITY_COL].unique():
    sub_train=df_train_all[df_train_all[ENTITY_COL]==la]
    sub_test =df_test[df_test[ENTITY_COL]==la]
    if sub_test.empty: 
        continue

    season=sub_train["stl_seasonal"].dropna().values
    if season.size == 0:
        continue
    if len(season) >= 12:
        base_pattern = season[-12:]
    else:
        base_pattern = season
    reps=int(np.ceil(len(sub_test)/len(base_pattern)))
    df_test.loc[sub_test.index,"stl_seasonal"]=np.tile(base_pattern,reps)[:len(sub_test)]

    t=sub_train["stl_trend"].dropna().values
    if len(t) >= 2:
        coef=np.polyfit(np.arange(len(t)),t,1)
        df_test.loc[sub_test.index,"stl_trend"]=coef[0]*np.arange(len(t),len(t)+len(sub_test))+coef[1]
    else:
        df_test.loc[sub_test.index,"stl_trend"]=t[-1] if len(t)>0 else np.nan

# Combine & create lags
combined=pd.concat([df_train_all,df_test]).sort_values([ENTITY_COL,TIME_COL])
for lag in best_lag_set:
    for comp in ["stl_trend","stl_seasonal","stl_resid"]:
        combined[f"{comp}_lag{lag}"] = combined.groupby(ENTITY_COL)[comp].shift(lag)

lag_cols=[f"{c}_lag{l}" for c in ["stl_trend","stl_seasonal","stl_resid"] for l in best_lag_set]
feature_cols=continuous_cols+categorical_cols+lag_cols

df_train=combined[combined[TIME_COL]<=TRAIN_END_DATE].dropna(subset=feature_cols)
df_test =combined[combined[TIME_COL]>=TEST_START_DATE].dropna(subset=feature_cols)

# =========================================================
# SCALE X AND y
# =========================================================
X_train=df_train[feature_cols].copy()
X_test =df_test[feature_cols].copy()

scaler=StandardScaler()
X_train[continuous_cols+lag_cols]=scaler.fit_transform(X_train[continuous_cols+lag_cols])
X_test[continuous_cols+lag_cols] =scaler.transform(X_test[continuous_cols+lag_cols])

y_train_raw=df_train[TARGET_COL].values.reshape(-1,1)
y_test_true=df_test[TARGET_COL].values

y_scaler=StandardScaler()
y_train_scaled=y_scaler.fit_transform(y_train_raw).ravel()
y_test_scaled =y_scaler.transform(y_test_true.reshape(-1,1)).ravel()

## Final fit

In [8]:
spatial_idx, local_idx, macro_idx, lag_idx = build_group_indices(feature_cols)
group_indices = [spatial_idx, local_idx, macro_idx, lag_idx]

train_groups = build_groups_tensors(X_train, group_indices)
test_groups  = build_groups_tensors(X_test,  group_indices)

group_dims = [len(idx) for idx in group_indices]
output_dims = [
    RFF_DIMS["spatial"],
    RFF_DIMS["local"],
    RFF_DIMS["macro"],
    RFF_DIMS["lag"],
]
gammas = [
    best_params["gamma_spatial"],
    best_params["gamma_local"],
    best_params["gamma_macro"],
    best_params["gamma_lag"],
]

model = RFFKernelRegressor(
    group_dims=group_dims,
    output_dims=output_dims,
    gammas=gammas,
    hidden_dim=best_params["hidden_dim"],
).to(device)

# =========================================================
# Final fit (train on full train, early stopping on test)
# =========================================================
y_pred_scaled = train_rff_model(
    model,
    train_groups,
    y_train_scaled,
    test_groups,
    y_test_scaled,
    epochs=80,
    lr=1e-3,
    weight_decay=best_params["weight_decay"],
    patience=8,
)

y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).ravel()

df_test["y_pred"]=y_pred
df_test["resid"]=y_test_true-y_pred

## Evaluation

In [9]:
# =========================================================
# UNCERTAINTY (PICP, PIW, CRPS)
# =========================================================
with torch.no_grad():
    train_pred_scaled = model(train_groups).cpu().numpy()
train_pred = y_scaler.inverse_transform(train_pred_scaled.reshape(-1,1)).ravel()
train_resid = y_train_raw.ravel() - train_pred
sigma_hat = np.std(train_resid)

y_std = np.full_like(y_pred, sigma_hat)

z = 1.96
y_lower = y_pred - z*y_std
y_upper = y_pred + z*y_std

PICP = np.mean((y_test_true >= y_lower) & (y_test_true <= y_upper))
PIW  = np.mean(y_upper - y_lower)
CRPS = crps_gaussian(y_test_true, y_pred, y_std)

# =========================================================
# GLOBAL METRICS
# =========================================================
global_mae   = mae(y_test_true,y_pred)
global_rmse  = rmse(y_test_true,y_pred)
global_smape = smape(y_test_true,y_pred)
global_mase  = mase(y_test_true,y_pred,y_train_raw.ravel())

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae=df_test.groupby(ENTITY_COL).apply(lambda x: mae(x[TARGET_COL],x["y_pred"]))
median_mae=np.median(la_mae)
p75_mae=np.percentile(la_mae,75)

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
la_resid = df_test.groupby(ENTITY_COL)["resid"].mean()

centroids = (
    df_test.drop_duplicates(ENTITY_COL)
           .set_index(ENTITY_COL)[["centroid_x","centroid_y"]]
           .loc[la_resid.index]   # <-- critical alignment
)

I_moran = morans_i(
    la_resid.values,
    centroids["centroid_x"].values,
    centroids["centroid_y"].values
)

monthly_resid=df_test.groupby(TIME_COL)["resid"].mean()
lb_res=acorr_ljungbox(monthly_resid,lags=[12],return_df=True)
q_stat=float(lb_res["lb_stat"].iloc[0])
p_val =float(lb_res["lb_pvalue"].iloc[0])

# =========================================================
# DIRECTION & GROWTH
# =========================================================
dir_acc = directional_accuracy(df_test,ENTITY_COL,TIME_COL,TARGET_COL,"y_pred")
gre_mae = growth_rate_error(df_test,ENTITY_COL,TIME_COL,TARGET_COL,"y_pred")

# =========================================================
# SAVE RESULTS
# =========================================================
summary_df=pd.DataFrame([{
    "model":"SparseGP_RFF",
    "lag_set":str(best_lag_set),
    "params":str(best_params),
    "MAE":global_mae,
    "RMSE":global_rmse,
    "sMAPE":global_smape,
    "MASE":global_mase,
    "Median_LA_MAE":median_mae,
    "P75_LA_MAE":p75_mae,
    "Morans_I":I_moran,
    "LjungBox_Q12":q_stat,
    "LjungBox_p":p_val,
    "Directional_Accuracy":dir_acc,
    "GrowthRateError_MAE":gre_mae,
    "PICP_95":PICP,
    "PIW_95":PIW,
    "CRPS":CRPS
}])

summary_df.to_excel("../../results/svm_rffkernel_final_test_results.xlsx",index=False)
la_mae.reset_index().to_excel("../../results/svm_rffkernel_la_mae.xlsx",index=False)

print("\n=== FINAL SPARSE GP TEST RESULTS SAVED ===")
print(summary_df.T)


C:\Users\slong\AppData\Local\Temp\ipykernel_2368\3573965944.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  la_mae=df_test.groupby(ENTITY_COL).apply(lambda x: mae(x[TARGET_COL],x["y_pred"]))
C:\Users\slong\AppData\Local\Temp\ipykernel_2368\1409322667.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(entity).apply(f).mean()
C:\Users\slong\AppData\Local\Temp\ipykernel_2368\1409322667.py:18: Fu


=== FINAL SPARSE GP TEST RESULTS SAVED ===
                                                                      0
model                                                      SparseGP_RFF
lag_set                                                   [1, 2, 3, 12]
params                {'gamma_spatial': 1.0, 'gamma_local': 0.1, 'ga...
MAE                                                        52115.439523
RMSE                                                       79219.817686
sMAPE                                                         17.212328
MASE                                                           2.665246
Median_LA_MAE                                              43981.459201
P75_LA_MAE                                                 57185.853516
Morans_I                                                       0.350311
LjungBox_Q12                                                 155.385991
LjungBox_p                                                          0.0
Directional_Accuracy